# Checkpoint 1 Extended EDA and Feature Engineering

Project: Predicting Student Dropout Risk in Online Learning Platforms  
Dataset: Open University Learning Analytics Dataset (OULAD)

Target definition used throughout:

```text
final_result == "Withdrawn" -> target_dropout = 1
Pass / Fail / Distinction -> target_dropout = 0
```

Prediction timing:

```text
Only data available by 25% of each module presentation is used.
```


In [1]:
# Dependency check
import importlib.util
import sys
from pathlib import Path

required_core = ["pandas", "numpy"]
optional = ["matplotlib", "seaborn", "sklearn"]

availability = {pkg: importlib.util.find_spec(pkg) is not None for pkg in required_core + optional}
print("Package availability:")
print(availability)

missing_core = [pkg for pkg in required_core if not availability[pkg]]
if missing_core:
    raise ImportError(f"Missing required packages: {missing_core}")

missing_optional = [pkg for pkg in optional if not availability[pkg]]
if missing_optional:
    print("\nOptional packages missing:", missing_optional)
    print("Install them in the active notebook environment if you want plots/model training:")
    print(f"{sys.executable} -m pip install matplotlib seaborn scikit-learn")


Package availability:
{'pandas': True, 'numpy': True, 'matplotlib': True, 'seaborn': True, 'sklearn': True}


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

DATA_DIR = Path("open+university+learning+analytics+dataset")
assert DATA_DIR.exists(), f"Dataset folder not found: {DATA_DIR.resolve()}"

def read_csv(name):
    return pd.read_csv(DATA_DIR / name, na_values=["?"])

assessments = read_csv("assessments.csv")
courses = read_csv("courses.csv")
student_assessment = read_csv("studentAssessment.csv")
student_info = read_csv("studentInfo.csv")
student_registration = read_csv("studentRegistration.csv")
vle = read_csv("vle.csv")

KEY = ["code_module", "code_presentation", "id_student"]
COURSE_KEY = ["code_module", "code_presentation"]

print("Loaded files:")
for name, df in {
    "assessments": assessments,
    "courses": courses,
    "student_assessment": student_assessment,
    "student_info": student_info,
    "student_registration": student_registration,
    "vle": vle,
}.items():
    print(f"{name:22s} {df.shape}")


Loaded files:
assessments            (206, 6)
courses                (22, 3)
student_assessment     (173912, 5)
student_info           (32593, 12)
student_registration   (32593, 5)
vle                    (6364, 6)


## 1. Relational Dataset Structure

OULAD is not a single flat file. The final model must be built by joining and aggregating multiple tables.

Main modeling unit:

```text
code_module + code_presentation + id_student
```

Lookup/reference keys:

- `courses.csv`: `code_module + code_presentation`
- `assessments.csv`: `id_assessment`, plus course keys
- `vle.csv`: `id_site + code_module + code_presentation`

Many-to-one tables:

- `studentAssessment.csv`: many assessment rows per student-course
- `studentVle.csv`: many clickstream rows per student-course


In [3]:
def summarize_df(df, name):
    return pd.Series({
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": int(df.isna().sum().sum()),
    }, name=name)

summary = pd.DataFrame([
    summarize_df(assessments, "assessments"),
    summarize_df(courses, "courses"),
    summarize_df(student_assessment, "studentAssessment"),
    summarize_df(student_info, "studentInfo"),
    summarize_df(student_registration, "studentRegistration"),
    summarize_df(vle, "vle"),
])
summary


,rows,columns,duplicate_rows,missing_cells
assessments,206,6,0,11
courses,22,3,0,0
studentAssessment,173912,5,0,173
studentInfo,32593,12,0,1111
studentRegistration,32593,5,0,22566
vle,6364,6,0,10486


In [4]:
print("Missing values by table:")
for name, df in {
    "assessments": assessments,
    "courses": courses,
    "studentAssessment": student_assessment,
    "studentInfo": student_info,
    "studentRegistration": student_registration,
    "vle": vle,
}.items():
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f"\n{name}")
    print(miss if len(miss) else "No missing values")


Missing values by table:

assessments
date    11
dtype: int64

courses
No missing values

studentAssessment
score    173
dtype: int64

studentInfo
imd_band    1111
dtype: int64

studentRegistration
date_unregistration    22521
date_registration         45
dtype: int64

vle
week_from    5243
week_to      5243
dtype: int64


## 2. Target Variable and 25% Checkpoint

The base table starts from `studentInfo.csv`. We add:

- `target_dropout`
- module presentation length
- checkpoint day at 25% of course length

This checkpoint is the leakage boundary. VLE rows and assessment evidence after this day must not be used.


In [5]:
base = student_info.copy()
base["target_dropout"] = (base["final_result"] == "Withdrawn").astype(int)

base = base.merge(courses, on=COURSE_KEY, how="left")
base["checkpoint_day"] = np.floor(base["module_presentation_length"] * 0.25).astype(int)

print("Base shape:", base.shape)
print("\nFinal result distribution:")
display(base["final_result"].value_counts())
print("\nBinary target distribution:")
display(base["target_dropout"].value_counts().rename(index={0: "not_withdrawn", 1: "withdrawn"}))
print("\nDropout rate:", round(base["target_dropout"].mean(), 4))
base.head()


Base shape: (32593, 15)

Final result distribution:


final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64


Binary target distribution:


target_dropout
not_withdrawn    22437
withdrawn        10156
Name: count, dtype: int64


Dropout rate: 0.3116


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,target_dropout,module_presentation_length,checkpoint_day
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,0,268,67
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,0,268,67
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,1,268,67
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,0,268,67
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,0,268,67


## 3. Extended EDA: Candidate Features vs Dropout

This section checks how the available pre-course/static variables relate to withdrawal.

They are screening views to guide feature engineering and fairness checks.


In [6]:
def dropout_rate_table(df, col, min_count=25):
    out = (
        df.groupby(col, dropna=False)
        .agg(
            records=("target_dropout", "size"),
            dropout_rate=("target_dropout", "mean"),
            withdrawn=("target_dropout", "sum"),
        )
        .reset_index()
        .sort_values("dropout_rate", ascending=False)
    )
    return out[out["records"] >= min_count]

categorical_cols = [
    "code_module", "code_presentation", "gender", "region", "highest_education",
    "imd_band", "age_band", "disability"
]

for col in categorical_cols:
    print(f"\nDropout rate by {col}")
    display(dropout_rate_table(base, col).head(20))



Dropout rate by code_module


,code_module,records,dropout_rate,withdrawn
2,CCC,4434,0.445422,1975
3,DDD,6272,0.358737,2250
5,FFF,7762,0.309585,2403
1,BBB,7909,0.301935,2388
4,EEE,2934,0.246080,722
0,AAA,748,0.168449,126
6,GGG,2534,0.115233,292



Dropout rate by code_presentation


,code_presentation,records,dropout_rate,withdrawn
3,2014J,11260,0.339787,3826
2,2014B,7804,0.334828,2613
0,2013B,4684,0.287788,1348
1,2013J,8845,0.267835,2369



Dropout rate by gender


...

,imd_band,records,dropout_rate,withdrawn
0,0-10%,3311,0.371791,1231
2,20-30%,3654,0.361522,1321
1,10-20,3516,0.354380,1246
4,40-50%,3256,0.320025,1042
3,30-40%,3539,0.309409,1095
6,60-70%,2905,0.295697,859
5,50-60%,3124,0.287772,899
8,80-90%,2762,0.280232,774
7,70-80%,2879,0.276832,797
9,90-100%,2536,0.258675,656



Dropout rate by age_band


,age_band,records,dropout_rate,withdrawn
0,0-35,22944,0.321696,7381
1,35-55,9433,0.288455,2721
2,55<=,216,0.250000,54



Dropout rate by disability


,disability,records,dropout_rate,withdrawn
1,Y,3164,0.393489,1245
0,N,29429,0.302797,8911


In [7]:
numeric_screen_cols = [
    "num_of_prev_attempts", "studied_credits",
    "module_presentation_length", "checkpoint_day"
]

print("Numeric feature summaries by target:")
display(base.groupby("target_dropout")[numeric_screen_cols].agg(["count", "mean", "median", "std"]).T)

print("\nCorrelation with target_dropout:")
display(base[numeric_screen_cols + ["target_dropout"]].corr(numeric_only=True)["target_dropout"].sort_values(ascending=False))


Numeric feature summaries by target:


target_dropout                                0             1
num_of_prev_attempts       count   22437.000000  10156.000000
                           mean        0.152605      0.186688
                           median      0.000000      0.000000
                           std         0.463091      0.513917
studied_credits            count   22437.000000  10156.000000
                           mean       74.475643     91.430189
                           median     60.000000     60.000000
                           std        36.811006     47.141690
module_presentation_length count   22437.000000  10156.000000
                           mean      256.060525    255.898779
                           median    262.000000    262.000000
                           std        13.163521     13.213350
checkpoint_day             count   22437.000000  10156.000000
                           mean       63.807996     63.751378
                           median     65.000000     65.000000
                           std         3.301288      3.308513


Correlation with target_dropout:


target_dropout                1.000000
studied_credits               0.191191
num_of_prev_attempts          0.032903
module_presentation_length   -0.005684
checkpoint_day               -0.007938
Name: target_dropout, dtype: float64

## 4. Registration Feature Engineering

Use `date_registration` because it is known before/near course start.

Do not use `date_unregistration` as a predictor. It directly reveals withdrawal timing and causes leakage.


In [8]:
registration_features = student_registration[KEY + ["date_registration"]].copy()
registration_features["registration_missing_flag"] = registration_features["date_registration"].isna().astype(int)
registration_features["date_registration"] = registration_features["date_registration"].fillna(
    registration_features["date_registration"].median()
)
registration_features["registered_before_start_days"] = (-registration_features["date_registration"]).clip(lower=0)
registration_features["registered_after_start_flag"] = (registration_features["date_registration"] > 0).astype(int)

registration_features.head()


,code_module,code_presentation,id_student,date_registration,registration_missing_flag,registered_before_start_days,registered_after_start_flag
0,AAA,2013J,11391,-159.0,0,159.0,0
1,AAA,2013J,28400,-53.0,0,53.0,0
2,AAA,2013J,30268,-92.0,0,92.0,0
3,AAA,2013J,31604,-52.0,0,52.0,0
4,AAA,2013J,32885,-176.0,0,176.0,0


## 5. VLE Feature Engineering up to 25% Checkpoint

The VLE table is large, so the feature generation below reads `studentVle.csv` in chunks.

Features produced:

- total early clicks
- number of VLE log rows/events
- active days
- distinct resources
- last activity day before checkpoint
- days since last activity at checkpoint
- click totals by major activity type


In [9]:
checkpoint_lookup = base[KEY + ["checkpoint_day"]].copy()
vle_meta = vle[["id_site", "code_module", "code_presentation", "activity_type"]].copy()

top_activity_types = (
    vle["activity_type"]
    .value_counts(dropna=False)
    .head(12)
    .index
    .tolist()
)
print("Top activity types retained:", top_activity_types)

event_parts = []
last_activity_parts = []
active_day_parts = []
distinct_site_parts = []
activity_parts = []

student_vle_path = DATA_DIR / "studentVle.csv"
usecols = ["code_module", "code_presentation", "id_student", "id_site", "date", "sum_click"]

for chunk in pd.read_csv(student_vle_path, usecols=usecols, chunksize=1_000_000):
    chunk = chunk.merge(checkpoint_lookup, on=KEY, how="inner")
    chunk = chunk[chunk["date"] <= chunk["checkpoint_day"]]
    if chunk.empty:
        continue

    event_agg = (
        chunk.groupby(KEY)
        .agg(
            total_clicks_25pct=("sum_click", "sum"),
            vle_events_25pct=("sum_click", "size"),
        )
        .reset_index()
    )
    event_parts.append(event_agg)

    last_activity_parts.append(
        chunk.groupby(KEY, as_index=False)["date"]
        .max()
        .rename(columns={"date": "last_activity_day_25pct"})
    )
    active_day_parts.append(chunk[KEY + ["date"]].drop_duplicates())
    distinct_site_parts.append(chunk[KEY + ["id_site"]].drop_duplicates())

    chunk = chunk.merge(vle_meta, on=["id_site", "code_module", "code_presentation"], how="left")
    chunk["activity_type"] = chunk["activity_type"].where(
        chunk["activity_type"].isin(top_activity_types), "other_activity"
    )
    activity_agg = (
        chunk.groupby(KEY + ["activity_type"])["sum_click"]
        .sum()
        .reset_index()
        .pivot_table(index=KEY, columns="activity_type", values="sum_click", fill_value=0)
        .reset_index()
    )
    activity_parts.append(activity_agg)

vle_events = pd.concat(event_parts, ignore_index=True).groupby(KEY, as_index=False).sum()
vle_last_activity = pd.concat(last_activity_parts, ignore_index=True).groupby(KEY, as_index=False).max()
vle_active_days = (
    pd.concat(active_day_parts, ignore_index=True)
    .drop_duplicates()
    .groupby(KEY)
    .size()
    .reset_index(name="active_days_25pct")
)
vle_distinct_sites = (
    pd.concat(distinct_site_parts, ignore_index=True)
    .drop_duplicates()
    .groupby(KEY)
    .size()
    .reset_index(name="distinct_sites_25pct")
)
vle_base = (
    vle_events
    .merge(vle_active_days, on=KEY, how="left")
    .merge(vle_distinct_sites, on=KEY, how="left")
    .merge(vle_last_activity, on=KEY, how="left")
)
vle_activity = pd.concat(activity_parts, ignore_index=True).groupby(KEY, as_index=False).sum()

vle_features = vle_base.merge(vle_activity, on=KEY, how="left")
vle_features = vle_features.merge(checkpoint_lookup, on=KEY, how="left")
vle_features["days_since_last_activity_at_checkpoint"] = (
    vle_features["checkpoint_day"] - vle_features["last_activity_day_25pct"]
)
vle_features = vle_features.drop(columns=["checkpoint_day"])

rename_map = {
    col: f"clicks_{str(col).replace('-', '_').replace(' ', '_')}_25pct"
    for col in vle_activity.columns
    if col not in KEY
}
vle_features = vle_features.rename(columns=rename_map)

print("VLE features shape:", vle_features.shape)
vle_features.head()


Top activity types retained: ['resource', 'subpage', 'oucontent', 'url', 'forumng', 'quiz', 'page', 'oucollaborate', 'questionnaire', 'ouwiki', 'dataplus', 'externalquiz']
VLE features shape: (29119, 22)


,code_module,code_presentation,id_student,total_clicks_25pct,vle_events_25pct,active_days_25pct,distinct_sites_25pct,last_activity_day_25pct,clicks_dataplus_25pct,clicks_forumng_25pct,...,clicks_oucontent_25pct,clicks_quiz_25pct,clicks_resource_25pct,clicks_subpage_25pct,clicks_url_25pct,clicks_externalquiz_25pct,clicks_ouwiki_25pct,clicks_page_25pct,clicks_questionnaire_25pct,days_since_last_activity_at_checkpoint
0,AAA,2013J,11391,545,92,19,34,65,0.0,98.0,...,344.0,0.0,9.0,23.0,1.0,0.0,0.0,0.0,0.0,2
1,AAA,2013J,28400,723,201,29,38,64,0.0,254.0,...,209.0,0.0,5.0,62.0,33.0,0.0,0.0,0.0,0.0,3
2,AAA,2013J,30268,281,76,12,22,12,0.0,126.0,...,66.0,0.0,4.0,22.0,4.0,0.0,0.0,0.0,0.0,55
3,AAA,2013J,31604,879,259,43,42,67,0.0,234.0,...,328.0,0.0,10.0,78.0,39.0,0.0,0.0,0.0,0.0,0
4,AAA,2013J,32885,608,163,29,35,67,0.0,138.0,...,315.0,0.0,7.0,29.0,6.0,0.0,0.0,0.0,0.0,0


## 6. Assessment Feature Engineering up to 25% Checkpoint

Use only assessments due by the checkpoint and submissions made by the checkpoint.

Students who have an early assessment due but no submission by the checkpoint get counted as missing for that early assessment.


In [10]:
assessment_defs = assessments.merge(
    courses.assign(checkpoint_day=np.floor(courses["module_presentation_length"] * 0.25).astype(int)),
    on=COURSE_KEY,
    how="left"
)
assessment_defs["date_missing_flag"] = assessment_defs["date"].isna().astype(int)
early_defs = assessment_defs[
    assessment_defs["date"].notna() & (assessment_defs["date"] <= assessment_defs["checkpoint_day"])
].copy()

early_due = (
    early_defs.groupby(COURSE_KEY)
    .agg(
        early_assessments_due_count=("id_assessment", "nunique"),
        early_assessment_weight_total=("weight", "sum"),
    )
    .reset_index()
)

early_submissions = student_assessment.merge(
    early_defs[["id_assessment", "code_module", "code_presentation", "assessment_type", "date", "weight", "checkpoint_day"]],
    on="id_assessment",
    how="inner"
)
early_submissions = early_submissions[early_submissions["date_submitted"] <= early_submissions["checkpoint_day"]].copy()
early_submissions["submission_delay"] = early_submissions["date_submitted"] - early_submissions["date"]
early_submissions["late_submission_flag"] = (early_submissions["submission_delay"] > 0).astype(int)
early_submissions["weighted_score_component"] = early_submissions["score"] * early_submissions["weight"]

assessment_features = (
    early_submissions.groupby(KEY)
    .agg(
        early_assessments_submitted_count=("id_assessment", "nunique"),
        mean_early_score=("score", "mean"),
        min_early_score=("score", "min"),
        max_early_score=("score", "max"),
        early_score_missing_count=("score", lambda s: s.isna().sum()),
        weighted_early_score_num=("weighted_score_component", "sum"),
        submitted_early_weight_total=("weight", "sum"),
        average_submission_delay=("submission_delay", "mean"),
        late_submission_count=("late_submission_flag", "sum"),
        banked_assessment_count=("is_banked", "sum"),
    )
    .reset_index()
)
assessment_features["weighted_early_score"] = np.where(
    assessment_features["submitted_early_weight_total"] > 0,
    assessment_features["weighted_early_score_num"] / assessment_features["submitted_early_weight_total"],
    np.nan,
)
assessment_features = assessment_features.drop(columns=["weighted_early_score_num"])

assessment_features = assessment_features.merge(base[KEY].drop_duplicates(), on=KEY, how="right")
assessment_features = assessment_features.merge(early_due, on=COURSE_KEY, how="left")
assessment_features["early_assessments_due_count"] = assessment_features["early_assessments_due_count"].fillna(0)
assessment_features["early_assessment_weight_total"] = assessment_features["early_assessment_weight_total"].fillna(0)
assessment_features["early_assessments_submitted_count"] = assessment_features["early_assessments_submitted_count"].fillna(0)
assessment_features["early_assessments_missing_count"] = (
    assessment_features["early_assessments_due_count"] - assessment_features["early_assessments_submitted_count"]
).clip(lower=0)

print("Assessment features shape:", assessment_features.shape)
assessment_features.head()


Assessment features shape: (32593, 16)


,code_module,code_presentation,id_student,early_assessments_submitted_count,mean_early_score,min_early_score,max_early_score,early_score_missing_count,submitted_early_weight_total,average_submission_delay,late_submission_count,banked_assessment_count,weighted_early_score,early_assessments_due_count,early_assessment_weight_total,early_assessments_missing_count
0,AAA,2013J,11391,2.0,81.5,78.0,85.0,0.0,30.0,-1.0,0.0,0.0,82.666667,2.0,30.0,0.0
1,AAA,2013J,28400,2.0,69.0,68.0,70.0,0.0,30.0,0.5,1.0,0.0,68.666667,2.0,30.0,0.0
2,AAA,2013J,30268,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,30.0,2.0
3,AAA,2013J,31604,2.0,71.5,71.0,72.0,0.0,30.0,-2.5,0.0,0.0,71.333333,2.0,30.0,0.0
4,AAA,2013J,32885,1.0,69.0,69.0,69.0,0.0,10.0,7.0,1.0,0.0,69.000000,2.0,30.0,1.0


## 7. Final Modeling Dataset

This is the dataset to use for Checkpoint 1 baseline modeling.

One row = one student-course enrollment.


In [11]:
profile_cols = [
    "code_module", "code_presentation", "id_student",
    "gender", "region", "highest_education", "imd_band", "age_band",
    "num_of_prev_attempts", "studied_credits", "disability",
    "module_presentation_length", "checkpoint_day", "target_dropout"
]

final_modeling_dataset = base[profile_cols].copy()
final_modeling_dataset["imd_band"] = final_modeling_dataset["imd_band"].fillna("Unknown")

final_modeling_dataset = final_modeling_dataset.merge(registration_features, on=KEY, how="left")
final_modeling_dataset = final_modeling_dataset.merge(vle_features, on=KEY, how="left")
final_modeling_dataset = final_modeling_dataset.merge(assessment_features, on=KEY, how="left")

zero_fill_prefixes = [
    "total_clicks", "vle_events", "active_days", "distinct_sites", "days_since",
    "clicks_", "early_assessments", "late_submission", "banked_assessment",
    "early_score_missing", "early_assessment_weight", "submitted_early_weight"
]
for col in final_modeling_dataset.columns:
    if any(col.startswith(prefix) for prefix in zero_fill_prefixes):
        final_modeling_dataset[col] = final_modeling_dataset[col].fillna(0)

for col in ["mean_early_score", "min_early_score", "max_early_score", "weighted_early_score", "average_submission_delay"]:
    if col in final_modeling_dataset.columns:
        final_modeling_dataset[f"{col}_missing_flag"] = final_modeling_dataset[col].isna().astype(int)
        final_modeling_dataset[col] = final_modeling_dataset[col].fillna(final_modeling_dataset[col].median())

print("Final modeling dataset shape:", final_modeling_dataset.shape)
print("Duplicate modeling keys:", final_modeling_dataset.duplicated(KEY).sum())
print("\nTarget distribution:")
display(final_modeling_dataset["target_dropout"].value_counts())
print("\nMissing values remaining:")
display(final_modeling_dataset.isna().sum().sort_values(ascending=False).head(20))
final_modeling_dataset.head()


Final modeling dataset shape: (32593, 55)
Duplicate modeling keys: 0

Target distribution:


target_dropout
0    22437
1    10156
Name: count, dtype: int64


Missing values remaining:


last_activity_day_25pct         3474
code_presentation                  0
code_module                        0
gender                             0
region                             0
highest_education                  0
id_student                         0
imd_band                           0
age_band                           0
studied_credits                    0
num_of_prev_attempts               0
module_presentation_length         0
checkpoint_day                     0
target_dropout                     0
disability                         0
date_registration                  0
registration_missing_flag          0
registered_after_start_flag        0
registered_before_start_days       0
total_clicks_25pct                 0
dtype: int64

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,banked_assessment_count,weighted_early_score,early_assessments_due_count,early_assessment_weight_total,early_assessments_missing_count,mean_early_score_missing_flag,min_early_score_missing_flag,max_early_score_missing_flag,weighted_early_score_missing_flag,average_submission_delay_missing_flag
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,82.666667,2.0,30.0,0.0,0,0,0,0,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,68.666667,2.0,30.0,0.0,0,0,0,0,0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,76.555556,2.0,30.0,2.0,1,1,1,1,1
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,71.333333,2.0,30.0,0.0,0,0,0,0,0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,69.000000,2.0,30.0,1.0,0,0,0,0,0


In [12]:
numeric_features = final_modeling_dataset.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [c for c in numeric_features if c not in ["id_student", "target_dropout"]]

categorical_features = [
    c for c in final_modeling_dataset.select_dtypes(include=["object"]).columns
    if c not in []
]

excluded_from_model = [
    "id_student",          # identifier only
    "target_dropout",     # target
    "final_result",        # original target; not in final dataset
    "date_unregistration", # leakage; not in final dataset
]

print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))
print("\nNumeric features:")
print(numeric_features)
print("\nCategorical features:")
print(categorical_features)
print("\nExplicitly excluded:")
print(excluded_from_model)


Numeric feature count: 45
Categorical feature count: 8

Numeric features:
['num_of_prev_attempts', 'studied_credits', 'module_presentation_length', 'checkpoint_day', 'date_registration', 'registration_missing_flag', 'registered_before_start_days', 'registered_after_start_flag', 'total_clicks_25pct', 'vle_events_25pct', 'active_days_25pct', 'distinct_sites_25pct', 'last_activity_day_25pct', 'clicks_dataplus_25pct', 'clicks_forumng_25pct', 'clicks_other_activity_25pct', 'clicks_oucollaborate_25pct', 'clicks_oucontent_25pct', 'clicks_quiz_25pct', 'clicks_resource_25pct', 'clicks_subpage_25pct', 'clicks_url_25pct', 'clicks_externalquiz_25pct', 'clicks_ouwiki_25pct', 'clicks_page_25pct', 'clicks_questionnaire_25pct', 'days_since_last_activity_at_checkpoint', 'early_assessments_submitted_count', 'mean_early_score', 'min_early_score', 'max_early_score', 'early_score_missing_count', 'submitted_early_weight_total', 'average_submission_delay', 'late_submission_count', 'banked_assessment_count', 

## 8. Correlation and Feature-Target Association

For numeric features, correlation with `target_dropout` gives a quick directional screen.

For categorical features, dropout-rate tables are usually more interpretable than raw correlation.


In [13]:
numeric_corr = (
    final_modeling_dataset[numeric_features + ["target_dropout"]]
    .corr(numeric_only=True)["target_dropout"]
    .drop("target_dropout")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
print("Top numeric associations with target_dropout:")
display(numeric_corr.head(25))

for col in categorical_features:
    print(f"\nDropout rate by {col}")
    display(dropout_rate_table(final_modeling_dataset, col).head(15))


Top numeric associations with target_dropout:


early_assessments_missing_count           0.562966
mean_early_score_missing_flag             0.494909
min_early_score_missing_flag              0.494909
max_early_score_missing_flag              0.494909
average_submission_delay_missing_flag     0.493743
early_assessments_submitted_count        -0.453045
last_activity_day_25pct                  -0.452113
submitted_early_weight_total             -0.440965
weighted_early_score_missing_flag         0.424970
active_days_25pct                        -0.375543
vle_events_25pct                         -0.313722
days_since_last_activity_at_checkpoint    0.310622
distinct_sites_25pct                     -0.310343
total_clicks_25pct                       -0.260140
clicks_other_activity_25pct              -0.236836
clicks_oucontent_25pct                   -0.213744
clicks_subpage_25pct                     -0.203019
studied_credits                           0.191191
clicks_forumng_25pct                     -0.177936
clicks_url_25pct               


Dropout rate by code_module


,code_module,records,dropout_rate,withdrawn
2,CCC,4434,0.445422,1975
3,DDD,6272,0.358737,2250
5,FFF,7762,0.309585,2403
1,BBB,7909,0.301935,2388
4,EEE,2934,0.246080,722
0,AAA,748,0.168449,126
6,GGG,2534,0.115233,292



Dropout rate by code_presentation


...

,imd_band,records,dropout_rate,withdrawn
0,0-10%,3311,0.371791,1231
2,20-30%,3654,0.361522,1321
1,10-20,3516,0.354380,1246
4,40-50%,3256,0.320025,1042
3,30-40%,3539,0.309409,1095
6,60-70%,2905,0.295697,859
5,50-60%,3124,0.287772,899
8,80-90%,2762,0.280232,774
7,70-80%,2879,0.276832,797
9,90-100%,2536,0.258675,656



Dropout rate by age_band


,age_band,records,dropout_rate,withdrawn
0,0-35,22944,0.321696,7381
1,35-55,9433,0.288455,2721
2,55<=,216,0.250000,54



Dropout rate by disability


,disability,records,dropout_rate,withdrawn
1,Y,3164,0.393489,1245
0,N,29429,0.302797,8911


## 9. Feature Decision: Columns to Use

Recommended feature groups:

1. Student profile: demographics, prior attempts, studied credits.
2. Registration timing: registration day, before/after start flags.
3. Course context: module, presentation, course length, checkpoint day.
4. Early VLE engagement: clicks, active days, resource breadth, activity-type clicks.
5. Early assessment behavior: submitted/missing counts, scores, delays, banked assessments.

Columns excluded because of leakage or identifier-only use:

- `final_result`
- `date_unregistration`
- post-checkpoint VLE activity
- post-checkpoint assessment submissions/scores
- `id_student` as a model input


## 10. Baseline ML Algorithm Comparison

Recommended algorithms:

- Logistic Regression: interpretable baseline; uses binary cross-entropy/log loss.
- Random Forest: nonlinear tabular baseline; handles interactions and mixed feature effects.
- HistGradientBoosting or Gradient Boosting: preferred candidate if it improves AUC/recall because boosting often performs well on tabular risk prediction.

The notebook attempts this comparison if `scikit-learn` is installed.


In [14]:
if not availability.get("sklearn", False):
    print("scikit-learn is not installed in this environment.")
    print("Install it, rerun from this cell, and the model comparison will execute.")
else:
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        average_precision_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.utils.class_weight import compute_sample_weight

    X = final_modeling_dataset[numeric_features + categorical_features].copy()
    y = final_modeling_dataset["target_dropout"].copy()

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe),
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

    models = {
        "LogisticRegression_balanced": LogisticRegression(
            max_iter=1000, class_weight="balanced", solver="lbfgs"
        ),
        "RandomForest_balanced": RandomForestClassifier(
            n_estimators=250, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
        ),
        "HistGradientBoosting_weighted": HistGradientBoostingClassifier(
            random_state=42, learning_rate=0.08, max_iter=200
        ),
    }

    results = []
    fitted_models = {}
    for name, model in models.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", model)])
        fit_kwargs = {}
        if name == "HistGradientBoosting_weighted":
            fit_kwargs["model__sample_weight"] = sample_weight
        pipe.fit(X_train, y_train, **fit_kwargs)

        if hasattr(pipe.named_steps["model"], "predict_proba"):
            scores = pipe.predict_proba(X_test)[:, 1]
        else:
            scores = pipe.decision_function(X_test)
        preds = (scores >= 0.5).astype(int)

        results.append({
            "model": name,
            "roc_auc": roc_auc_score(y_test, scores),
            "pr_auc": average_precision_score(y_test, scores),
            "precision_at_0_5": precision_score(y_test, preds, zero_division=0),
            "recall_at_0_5": recall_score(y_test, preds, zero_division=0),
            "f1_at_0_5": f1_score(y_test, preds, zero_division=0),
        })
        fitted_models[name] = pipe

    results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
    display(results_df)

    best_name = results_df.iloc[0]["model"]
    best_model = fitted_models[best_name]
    best_scores = best_model.predict_proba(X_test)[:, 1]
    best_preds = (best_scores >= 0.5).astype(int)
    print("Best model:", best_name)
    print("Confusion matrix at threshold 0.5:")
    print(confusion_matrix(y_test, best_preds))
    print(classification_report(y_test, best_preds, target_names=["not_withdrawn", "withdrawn"]))


,model,roc_auc,pr_auc,precision_at_0_5,recall_at_0_5,f1_at_0_5
2,HistGradientBoosting_weighted,0.881312,0.819950,0.720694,0.753052,0.736518
1,RandomForest_balanced,0.878461,0.807109,0.804241,0.642379,0.714254
0,LogisticRegression_balanced,0.871242,0.785758,0.707908,0.736904,0.722115


Best model: HistGradientBoosting_weighted
Confusion matrix at threshold 0.5:
[[4869  741]
 [ 627 1912]]
               precision    recall  f1-score   support

not_withdrawn       0.89      0.87      0.88      5610
    withdrawn       0.72      0.75      0.74      2539

     accuracy                           0.83      8149
    macro avg       0.80      0.81      0.81      8149
 weighted avg       0.83      0.83      0.83      8149



## 11. PCA Check

PCA is not automatically required.

Decision rule:

- If one-hot encoding creates a very large feature space, test PCA for Logistic Regression.
- Do not use PCA by default for tree-based models because PCA reduces interpretability and tree models can handle many tabular features.
- Compare model performance with and without PCA before deciding.


In [15]:
if not availability.get("sklearn", False):
    print("PCA check skipped because scikit-learn is not installed.")
else:
    from sklearn.decomposition import PCA

    X_encoded = preprocess.fit_transform(X)
    print("Encoded feature matrix shape:", X_encoded.shape)

    if X_encoded.shape[1] <= 50:
        print("Feature count is modest. PCA is not required by default.")
    else:
        pca_probe = PCA(n_components=0.95, random_state=42)
        pca_probe.fit(X_encoded)
        print("Original encoded feature count:", X_encoded.shape[1])
        print("Components needed for 95% variance:", pca_probe.n_components_)
        print("Recommendation: test Logistic Regression with PCA, but keep tree-based model without PCA.")


Encoded feature matrix shape: (32593, 92)
Original encoded feature count: 92
Components needed for 95% variance: 39
Recommendation: test Logistic Regression with PCA, but keep tree-based model without PCA.


## 12. Loss Function and Class Imbalance Recommendation

The positive class is `Withdrawn`. In this dataset, withdrawal is meaningful but not extremely rare.

Recommended loss/objective:

- Logistic Regression: binary cross-entropy, also called log loss.
- Tree/boosting classifiers: class-weighted split criteria or sample-weighted training where supported.

Recommended imbalance strategy:

- Use `class_weight="balanced"` for Logistic Regression and Random Forest.
- Use sample weights for HistGradientBoosting if chosen.
- Evaluate with ROC-AUC and PR-AUC.
- Report recall, precision, F1, and confusion matrix at the selected intervention threshold.

Why not accuracy alone?

Accuracy can look acceptable even when the model misses many withdrawn students. For this project, missing withdrawn students means missing opportunities for early support, so recall and PR-AUC are important.


## 13. Checkpoint 1 Conclusions After Running

Checkpoint 1 confirms that the OULAD data has enough early signal to support dropout-risk modeling at the 25% course checkpoint.

### 13.1 Strongest Feature Signals

The strongest numeric associations with dropout came from early assessment completion and early engagement behavior. The top signals were:

- `early_assessments_missing_count` had the strongest positive relationship with withdrawal (`corr = 0.563`).
- Missing early score indicators were also strong positive signals (`corr ≈ 0.495`).
- `early_assessments_submitted_count` was strongly negative (`corr = -0.453`), meaning students who submitted early assessments were less likely to withdraw.
- `last_activity_day_25pct` was strongly negative (`corr = -0.452`), while `days_since_last_activity_at_checkpoint` was positive (`corr = 0.311`). This means recent activity is protective, and inactivity near the checkpoint is risky.
- VLE engagement measures such as `active_days_25pct`, `vle_events_25pct`, `distinct_sites_25pct`, and `total_clicks_25pct` were all negatively correlated with withdrawal.
- `studied_credits` had a positive association with withdrawal (`corr = 0.191`), suggesting heavier study load may be related to dropout risk.

Categorical EDA also showed meaningful subgroup differences:

- Dropout rate was highest for module `CCC` (`44.54%`) and lowest for `GGG` (`11.52%`).
- Students with declared disability had a higher withdrawal rate (`39.35%`) than those without declared disability (`30.28%`).
- Lower IMD/deprivation bands had higher withdrawal rates, with `0-10%` at `37.18%` and `90-100%` at `25.87%`.
- Students with `No Formal quals` had a high withdrawal rate (`42.94%`), while `Post Graduate Qualification` was lower (`23.64%`).

### 13.2 Final Modeling Dataset

The final checkpoint-safe modeling dataset contains:

- `32,593` rows, one row per student-course enrollment.
- `55` columns after joining profile, registration, early VLE, and early assessment features.
- `0` duplicate modeling keys for `code_module + code_presentation + id_student`.
- Target distribution: `22,437` not withdrawn and `10,156` withdrawn.
- Withdrawal rate: `31.16%`.
- Feature split: `45` numeric features and `8` categorical features.

The notebook correctly excludes leakage columns from modeling:

- `final_result` is used only to create the target.
- `date_unregistration` is excluded because it directly reveals withdrawal timing.
- VLE and assessment records are filtered to the 25% checkpoint.
- `id_student` is excluded as an identifier-only field.

One remaining missing-value issue is `last_activity_day_25pct`, which is missing for `3,474` records. These are likely students with no early VLE activity. For Checkpoint 2, this should be converted into explicit features such as `has_early_vle_activity` and a safe filled value for last activity.

### 13.3 Baseline Model Results

The best baseline model was `HistGradientBoosting_weighted`.

| Model | ROC-AUC | PR-AUC | Precision @ 0.5 | Recall @ 0.5 | F1 @ 0.5 |
|---|---:|---:|---:|---:|---:|
| HistGradientBoosting_weighted | 0.881 | 0.820 | 0.721 | 0.753 | 0.737 |
| RandomForest_balanced | 0.878 | 0.807 | 0.804 | 0.642 | 0.714 |
| LogisticRegression_balanced | 0.871 | 0.786 | 0.708 | 0.737 | 0.722 |

At threshold `0.5`, the best model produced this confusion matrix:

```text
                 Predicted Not Withdrawn   Predicted Withdrawn
Actual Not Withdrawn          4869                  741
Actual Withdrawn               627                 1912
```

This gives withdrawn-class recall of `0.75`, meaning the model identified about three quarters of withdrawn students in the held-out test set at the default threshold. The ROC-AUC of `0.881` is above the project target of `0.78`, so Checkpoint 1 meets the expected predictive feasibility target.

### 13.4 PCA Decision

After preprocessing and one-hot encoding, the feature matrix had:

- `32,593` rows
- `92` encoded features

PCA required `39` components to retain `95%` variance.

Conclusion: PCA is not required by default. The encoded feature count is manageable, and the best-performing model is a tree-based boosting model where PCA would reduce interpretability. PCA may be tested only as an optional Logistic Regression experiment, not as the default modeling path.

### 13.5 Loss Function and Class Imbalance

The withdrawal class is moderately imbalanced, not extremely rare:

- Not withdrawn: `68.84%`
- Withdrawn: `31.16%`

The appropriate objective is binary probabilistic classification:

- Logistic Regression uses binary cross-entropy / log loss with `class_weight="balanced"`.
- Random Forest uses `class_weight="balanced_subsample"`.
- HistGradientBoosting uses balanced sample weights.

Accuracy alone should not be the main metric. The project should continue reporting ROC-AUC, PR-AUC, precision, recall, F1, and confusion matrix because the cost of missing withdrawn students is operationally important.

### 13.6 Checkpoint 2 Priorities

For Checkpoint 2, the next work should be:

- Fix the `last_activity_day_25pct` missing-value handling with an explicit no-early-activity indicator.
- Add hyperparameter tuning for HistGradientBoosting and Random Forest.
- Select an intervention threshold based on recall, precision, and advisor capacity instead of defaulting to `0.5`.
- Add fairness audit metrics by `gender`, `disability`, and `imd_band`.
- Add feature importance or explainability outputs so the model can be explained in the final report and demo.
- Save the final modeling dataset and trained pipeline for deployment testing.
